## Disease Symptoms Dataset (EDA)
- Goal: understand a symptoms -> disease dataset.
- Upload `disease_symptoms_raw.csv` to Colab first, then run the notebook.
- I split the raw data inside the notebook and save new train/test CSV files.
- Simple checks + a few plots + (optional) a tiny baseline model.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

raw_path = Path('disease_symptoms_raw.csv')

if not raw_path.exists():
    raw_path = Path('data/raw/disease_symptoms_raw.csv')

print('raw_path:', raw_path)

raw_df = pd.read_csv(raw_path)

raw_df.head()


### Clean columns
- I clean the column names first and then split the raw data.


In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    drop_cols = [c for c in df.columns if c == '' or c.lower().startswith('unnamed')]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

raw_df = clean_columns(raw_df)

print('Raw shape:', raw_df.shape)

raw_df.columns[-5:]


### Split raw data
- I split the raw file into train and test sets and save them as CSV files.


In [ ]:
from sklearn.model_selection import train_test_split

target = 'prognosis'

train_df, test_df = train_test_split(
    raw_df,
    test_size=0.2,
    random_state=42,
    stratify=raw_df[target]
)

output_dir = Path('data')
output_dir.mkdir(exist_ok=True)

train_output = output_dir / 'Training.csv'
test_output = output_dir / 'Testing.csv'

train_df.to_csv(train_output, index=False)
test_df.to_csv(test_output, index=False)

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Saved:', train_output)
print('Saved:', test_output)


### First look
- Quick look at the training set after the split.


In [ ]:
train_df.info()
train_df.head()


### Missing values
- Check if anything is missing.


In [ ]:
train_df.isna().sum().sort_values(ascending=False).head(10)


### Label balance
- How many diseases are in the training set, and how balanced are they?


In [ ]:
target = 'prognosis'
print('Unique diseases:', train_df[target].nunique())

vc = train_df[target].value_counts()
display(vc.head(10))

plt.figure(figsize=(10, 4))
vc.head(10).plot(kind='bar')
plt.title('Top 10 diseases (train)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Symptoms per patient
- Each row is a set of symptoms (0/1). I counted how many symptoms are marked as 1.


In [ ]:
symptom_cols = [c for c in train_df.columns if c != target]
train_df['symptom_count'] = train_df[symptom_cols].sum(axis=1)

display(train_df['symptom_count'].describe())

plt.figure(figsize=(8, 4))
sns.histplot(train_df['symptom_count'], bins=15)
plt.title('Symptoms per patient')
plt.tight_layout()
plt.show()


### Common symptoms
- Which symptoms appear most often in the training set?


In [ ]:
top_symptoms = train_df[symptom_cols].sum().sort_values(ascending=False).head(10)
display(top_symptoms)

plt.figure(figsize=(10, 4))
sns.barplot(x=top_symptoms.values, y=top_symptoms.index, color='steelblue')
plt.title('Top 10 symptoms (count of 1s)')
plt.xlabel('Count')
plt.ylabel('')
plt.tight_layout()
plt.show()


### Optional: tiny baseline model
- Not the final model, just a quick baseline to see if the data is learnable.


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score

X = train_df[symptom_cols]
y = train_df[target].astype(str)
X_test = test_df[symptom_cols]
y_test_text = test_df[target].astype(str)

le = LabelEncoder()
y_enc = le.fit_transform(y)
y_test = le.transform(y_test_text)

model = BernoulliNB()
model.fit(X, y_enc)

pred = model.predict(X_test)
print('Test accuracy:', round(accuracy_score(y_test, pred), 4))

sample = X_test.iloc[[0]]
proba = model.predict_proba(sample)[0]
top3 = np.argsort(proba)[::-1][:3]
print('Top-3 predictions for 1 sample:')
for i in top3:
    print('-', le.inverse_transform([i])[0], f'{proba[i]*100:.1f}%')
print('Saved split files are used as the project train/test sets.')


### Short findings
- Raw set: 4,962 rows, 132 symptom features, 41 diseases.
- I split the raw file into 3,978 training rows and 984 test rows.
- The split stays balanced: about 97-98 rows per disease in train and 24 in test.
- Symptoms per patient: mean 7.47, median 6.
- Most common symptoms in this file: fatigue, vomiting, high_fever, loss_of_appetite, nausea.
- The notebook also saves new `data/Training.csv` and `data/Testing.csv` files.
